# Predicting Hit Songs Using Repeated Chorus

Dhanush Sai Suprapadha - PES2UG24CS154

Deepthi V - PES2UG24CS150

This walkthrough reads validated experiment artifacts. It never retrains or tunes on historical labels.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT))
from chorus_hit.artifacts import load_run
from chorus_hit.evaluation import classification_metrics
from chorus_hit.reporting import summary_text
RUN_ID='v2_nested_001'
assets=load_run(RUN_ID)
summary=assets.summary
display(pd.Series({'run_id':RUN_ID,'model':summary['model_name'],'raw_features':summary['feature_count'],'evaluation':summary['evaluation_status'],'source_hash':assets.manifest['dataset']['source_sha256']}))

run_id                                              v2_nested_001
model                                                    lr_mi_50
raw_features                                                  518
evaluation                                     nested_development
source_hash     79951ffc393fc836405509326d1ee03efd96434327d800...
dtype: object

## 1. Dataset and task

Class 1 means a source year-end Billboard hit. Class 0 contains other chart songs. Both classes can have charted. Source recordings are unavailable. No artist/title/path or chart-position metadata enters the model.

In [2]:
display(assets.data[['track_id','artist','title','label']].head())
display(assets.data.label.value_counts().rename('songs'))
display(pd.Series(summary['counts']))

,track_id,artist,title,label
0,CH0001,The Weeknd,Blinding Lights,1
1,CH0002,Olivia Rodrigo,Good 4 U,1
2,CH0003,Olivia Rodrigo,Drivers License,1
3,CH0004,Lil Nas X,Montero (Call Me By Your Name),1
4,CH0005,BTS,Butter,1


label
0    385
1    366
Name: songs, dtype: int64

train_songs                  597
train_artists                 52
evaluation_songs             597
evaluation_artists            52
historical_songs_reserved    154
dtype: int64

## 2. Grouped model selection

All learned preprocessing fits inside the training folds. V2 uses a fixed candidate list and separate outer folds to assess the selection procedure. The historical partition never tunes settings. Artist strings are the declared identity level; aliases and collaborations remain unresolved.

In [3]:
display(assets.bundle['pipeline'])
config_path=assets.path/'config.json'
if config_path.exists():
    config=json.loads(config_path.read_text())
    display(pd.DataFrame(config['candidates']))
    display(pd.read_csv(assets.path/'final_development_ranking.csv').sort_values('mean_balanced_accuracy',ascending=False))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('variance', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](518,)","['chroma_stft_kew_0','chroma_stft_min_0','chroma_stft_max_0',..., 'zero_crossing_rate_mean_0','zero_crossing_rate_median_0', 'zero_crossing_rate_kurtosis_0']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,518
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constan

,id,family,params,representation,role
0,control_dummy,dummy,{},{'kind': 'none'},V1 preprocessing and V1 development-chosen set...
1,control_logistic,logistic,{'C': 0.01},"{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
2,control_lda,lda,{'shrinkage': 0.7},"{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
3,control_linear_svm,linear_svm,{'C': 0.1},"{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
4,control_rbf_svm,rbf_svm,"{'C': 1.0, 'gamma': 0.001}","{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
5,control_poly_svm,poly_svm,{'C': 1.0},"{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
6,control_random_forest,random_forest,"{'max_depth': None, 'min_samples_leaf': 5}",{'kind': 'none'},V1 preprocessing and V1 development-chosen set...
7,control_gradient_boosting,gradient_boosting,"{'learning_rate': 0.03, 'max_depth': 1}",{'kind': 'none'},V1 preprocessing and V1 development-chosen set...
8,control_mlp,mlp,{'alpha': 1.0},"{'kind': 'pca', 'value': 0.95}",V1 preprocessing and V1 development-chosen set...
9,lr_none_None,logistic,{'C': 0.1},{'kind': 'none'},paired representation ablation; fixed C=0.1


,stage,candidate_id,family,representation,eligible,mean_balanced_accuracy,fold_std,mean_train_balanced_accuracy,mean_dimensions,failed_folds,warning_count
17,final_development,lr_mi_50,logistic,"{""kind"": ""mi"", ""value"": 50}",True,0.528790,0.029265,0.651309,50.000000,0,0
31,final_development,mlp_pca30,mlp,"{""kind"": ""pca"", ""value"": 30}",True,0.519952,0.019264,0.999183,30.000000,0,0
27,final_development,poly_anova,poly_svm,"{""kind"": ""anova"", ""value"": 100}",True,0.516735,0.023513,0.839083,100.000000,0,0
22,final_development,linear_anova_c0p1,linear_svm,"{""kind"": ""anova"", ""value"": 100}",True,0.515845,0.023232,0.741879,100.000000,0,0
19,final_development,lr_none_c0p01,logistic,"{""kind"": ""none""}",True,0.514755,0.023902,0.832756,517.666667,0,0
14,final_development,lr_pca_0p95,logistic,"{""kind"": ""pca"", ""value"": 0.95}",True,0.514316,0.033120,0.802021,143.000000,0,0
15,final_development,lr_anova_30,logistic,"{""kind"": ""anova"", ""value"": 30}",True,0.510014,0.026055,0.644734,30.000000,0,0
34,final_development,extra_leaf10,extra_trees,"{""kind"": ""none""}",True,0.509786,0.028122,0.966596,517.666667,0,0
3,final_development,control_linear_svm,linear_svm,"{""kind"": ""pca"", ""value"": 0.95}",True,0.507602,0.038304,0.827523,143.000000,0,0
25,final_development,rbf_none_c10,rbf_svm,"{""kind"": ""none""}",True,0.506660,0.019784,1.000000,517.666667,0,0


## 3. Recompute the reported evaluation

Nested rows come from models fitted without their outer-fold groups. The final saved candidate is a separate fit on all development rows. Comparing its training predictions to these nested predictions would be the wrong check. Different outer candidates can have incomparable score scales, so pooled nested ROC-AUC is omitted.

In [4]:
predictions=assets.predictions
scores=None if summary['evaluation_status']=='nested_development' else predictions.score
recomputed=classification_metrics(predictions.label,predictions.prediction,scores)
for key in ['accuracy','balanced_accuracy','precision','recall','f1']:
    assert abs(recomputed[key]-summary['metrics'][key])<1e-12
display(pd.Series(recomputed))
display(pd.DataFrame(summary['target_status']['metrics']).T)
display(pd.DataFrame(summary['ci95'],index=['lower','upper']).T)

accuracy                             0.529313
balanced_accuracy                    0.528502
precision                            0.516014
recall                                    0.5
f1                                   0.507881
class_0_recall                       0.557003
macro_f1                             0.528419
roc_auc                                  None
confusion_matrix     [[171, 136], [145, 145]]
n                                         597
support                  {'0': 307, '1': 290}
undefined                                  []
dtype: object

,value,passed
accuracy,0.529313,False
balanced_accuracy,0.528502,False
precision,0.516014,False
recall,0.5,False
f1,0.507881,False


,lower,upper
accuracy,0.483265,0.573857
balanced_accuracy,0.482582,0.57345
precision,0.435963,0.594262
recall,0.436859,0.561339
f1,0.443493,0.563182
class_0_recall,0.49236,0.614148
macro_f1,0.481767,0.572543
roc_auc,None,None


## 4. Diagnostics

Training/validation gaps and learning curves describe this selected configuration. They are retrospective diagnostics, not a fresh evaluation or proof of a single performance bottleneck.

In [5]:
curve=assets.path/'learning_curves.csv'
if curve.exists():
    display(pd.read_csv(curve))
display(predictions.head(10))
print('All errors remain in the record:',int((predictions.label!=predictions.prediction).sum()))

,fold,stage,fit_rows,fit_groups,status,train_balanced_accuracy,validation_balanced_accuracy,error
0,0,learning_0.25,105,9,complete,0.809913,0.486728,NaN
1,0,learning_0.5,208,17,complete,0.671393,0.531812,NaN
2,0,learning_1.0,400,33,complete,0.662115,0.529023,NaN
3,1,learning_0.25,125,9,complete,0.760246,0.496990,NaN
4,1,learning_0.5,243,17,complete,0.710224,0.535388,NaN
5,1,learning_1.0,394,34,complete,0.625439,0.564515,NaN
6,2,learning_0.25,129,10,complete,0.875782,0.501186,NaN
7,2,learning_0.5,208,19,complete,0.721985,0.474319,NaN
8,2,learning_1.0,400,37,complete,0.666375,0.492832,NaN


,track_id,artist_group,label,prediction,score,score_kind,fold,candidate_id,family
0,CH0008,ed sheeran,1,0,-0.437245,decision_margin,0,lr_pca_30,logistic
1,CH0015,doja cat,1,1,0.518220,decision_margin,0,lr_pca_30,logistic
2,CH0018,doja cat,1,0,-1.437220,decision_margin,0,lr_pca_30,logistic
3,CH0026,maroon 5,1,0,-0.444978,decision_margin,0,lr_pca_30,logistic
4,CH0027,doja cat,1,0,-0.139345,decision_margin,0,lr_pca_30,logistic
5,CH0072,ed sheeran,1,0,-0.398524,decision_margin,0,lr_pca_30,logistic
6,CH0076,camila cabello,1,0,-0.424367,decision_margin,0,lr_pca_30,logistic
7,CH0087,maroon 5,1,0,-0.003052,decision_margin,0,lr_pca_30,logistic
8,CH0091,ed sheeran,1,0,-0.452981,decision_margin,0,lr_pca_30,logistic
9,CH0106,ed sheeran,1,0,-0.656992,decision_margin,0,lr_pca_30,logistic


All errors remain in the record: 281


## 5. Interpretation

lr_mi_50 records 52.9% balanced accuracy for nested development.

The approximate 95% group-bootstrap interval (48.3%-57.3%) includes 50%; a reliable advantage over chance is not established.

The five-metric target is not met in this evaluation. Fresh-test confirmation is unavailable.

In [6]:
text=summary_text(summary)
display(Markdown('**'+text['title']+'**\n\n'+text['method']+'\n\n'+text['result']+'\n\n'+text['uncertainty']+'\n\n'+text['conclusion']))

**lr_mi_50: nested development**

The model accepts 518 ordered raw audio features. Selection criterion: 3-fold development balanced accuracy; predeclared ID tie break.

lr_mi_50 records 52.9% balanced accuracy for nested development.

The approximate 95% group-bootstrap interval (48.3%-57.3%) includes 50%; a reliable advantage over chance is not established.

The five-metric target is not met in this evaluation. Fresh-test confirmation is unavailable.

## 6. Demonstration and missing inputs

Start the Streamlit app and inspect its run identity. The original demo remains active; the V2 candidate is a research option. Audio uploads use the versioned extractor, but waveform parity remains unverified.

New experiments require permitted recordings, verified label/recording identities and a new locked evaluation set. Embedding interfaces have only software-fixture tests, with no measured real-song embedding results.

Sources: [reference paper](https://cs229.stanford.edu/proj2021spr/report2/81974051.pdf), [public feature dataset](https://github.com/AntoniosMalak/Predicting-Hit-Songs-Using-Repeated-Chorus), and local immutable run manifests. Both students should run and understand the project before presenting it.